# 01 · Veri Hazırlama

**Proje:** Akıllı Gıda Etiketi Analizi
**Bu not defterinin amacı:** Open Food Facts veritabanından projenin veri kümesini (`dataset.parquet`) oluşturmak.

| | |
|---|---|
| **Girdi** | Open Food Facts CSV dökümü (~1,2 GB sıkıştırılmış, ~4,5 milyon ürün) |
| **Çıktı 1** | `off_slim.parquet` — tüm ürünler, yalnızca gerekli sütunlar |
| **Çıktı 2** | `dataset.parquet` — Türkiye ürünleri + 28.300 Avrupa ürünü |
| **Çıktı 3** | `reports/` altında rapora girecek özet tablolar |

**Akış:**
1. Kurulum ve ayarlar
2. Ham veriyi indirme
2.1 Bütün sütunlara bakış: hangi sütun ne kadar dolu?
3. CSV → Parquet (hafifletilmiş)
4. Türkiye ürünlerini işaretleme
5. Veri keşfi: etiket varyantları, ülke dağılımı, eleme tablosu
6. Proje veri kümesini oluşturma
7. Kontroller
8. Eski dosyaların temizliği

> Not defteri "Tümünü çalıştır" ile baştan sona çalışacak şekilde yazıldı. Uzun süren adımlar (indirme, dönüştürme), dosya zaten varsa otomatik olarak atlanır.

## 1. Kurulum ve ayarlar

- Google Drive bağlanır; bütün dosyalar Drive'da saklanır (Colab oturumu kapansa da kaybolmaz).
- **DuckDB** kurulur: büyük dosyaları belleğe yüklemeden SQL ile sorgulamamızı sağlayan, çok çekirdekli çalışan bir veritabanı.
- Bütün sabitler (dosya yolları, tohum değeri, örnek sayısı) tek yerde tanımlanır. Bir değeri değiştirmek gerekirse sadece bu hücre değişir.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q duckdb==1.5.6

import os
import duckdb
import pandas as pd

# --- Dosya yolları ---
BASE    = "/content/drive/MyDrive/off"
CSV     = f"{BASE}/products.csv.gz"        # ham veri
SLIM    = f"{BASE}/off_slim.parquet"       # hafifletilmiş tüm veri
DS      = f"{BASE}/dataset.parquet"        # proje veri kümesi
REPORTS = f"{BASE}/reports"                # rapora girecek tablolar
os.makedirs(REPORTS, exist_ok=True)

CSV_URL = "https://static.openfoodfacts.org/data/en.openfoodfacts.org.products.csv.gz"

# --- Deney ayarları ---
SEED       = 42          # tekrar üretilebilirlik için sabit tohum
N_AB       = 28_300      # Avrupa'dan seçilecek ürün sayısı
AB_ULKELER = ["france", "germany", "spain", "italy", "belgium", "netherlands"]

print("DuckDB sürümü:", duckdb.__version__)

## 2. Ham veriyi indirme

Open Food Facts her gece güncel bir CSV dökümü yayımlar. `wget -c` indirme yarıda kesilirse kaldığı yerden devam eder.

**Dosya zaten varsa bu adım atlanır.** Veri her gün güncellendiği için, raporda indirme tarihini belirtmek gerekir (tekrar üretilebilirlik).

In [ ]:
if os.path.exists(CSV):
    print(f"Zaten var, atlanıyor: {CSV} ({os.path.getsize(CSV)/1e9:.2f} GB)")
else:
    !wget -c -O "{CSV}" "{CSV_URL}"

from datetime import datetime
print("Dosya tarihi:", datetime.fromtimestamp(os.path.getmtime(CSV)).strftime("%Y-%m-%d %H:%M"))

## 2.1 Bütün sütunlara bakış: hangi sütun ne kadar dolu?

CSV'de **211 sütun** var. Hangilerini kullanacağımıza karar vermeden önce her sütun için üç bilgiye bakıyoruz:

| Kolon | Anlamı |
|---|---|
| `tum_%` | Bütün ürünlerde bu sütunun dolu olma oranı |
| `tr_%` | Türkiye ürünlerinde (ülke etiketi veya 868/869 barkod) dolu olma oranı |
| `tr_ornek` | Bir Türkiye ürününden örnek değer: sütunun içinde ne olduğunu görmek için |

**Neden önemli?** Adı çok kullanışlı görünen bir sütun Türkiye ürünlerinde %2 doluysa işe yaramaz. Örneğin `origins` (menşe) ya da `stores` (mağaza) sütunları ilginç ama genelde çok boş.

**Dikkat:** `unknown` gibi değerler de "dolu" sayılır. Örneğin `nutriscore_grade` neredeyse her üründe dolu görünür, ama büyük kısmı `unknown`'dur. Bir sütunu seçmeden önce örnek değerine de bakmak bu yüzden önemli.

**Nasıl hesaplanıyor?** 211 sütunun her biri için `count(sütun)` (boş olmayan değer sayısı) tek bir SQL sorgusunda hesaplanır; dosya bir kez taranır. Sonuç `reports/sutun_doluluk.csv` olarak kaydedilir ve bir sonraki çalıştırmada yeniden hesaplanmaz.

> CSV'den okunduğu için birkaç dakika sürebilir. Daha önce oluşturulmuş, bütün sütunları içeren `products.parquet` Drive'da varsa o kullanılır; çok daha hızlıdır.

In [ ]:
DOLULUK = f"{REPORTS}/sutun_doluluk.csv"

if os.path.exists(DOLULUK):
    print("Önceden hesaplanmış, okunuyor:", DOLULUK)
    doluluk = pd.read_csv(DOLULUK)
else:
    FULL = f"{BASE}/products.parquet"
    if os.path.exists(FULL):
        kaynak = f"'{FULL}'"
    else:
        kaynak = (f"read_csv('{CSV}', delim='\t', quote='', header=true, "
                  f"all_varchar=true, ignore_errors=true)")

    sutunlar = duckdb.sql(f"DESCRIBE SELECT * FROM {kaynak}").df()["column_name"].tolist()
    print("Sütun sayısı:", len(sutunlar))

    TR = ("(regexp_matches(coalesce(countries_tags,''), 'en:(turkey|turkiye)') "
          "OR regexp_matches(coalesce(code,''), '^86[89]'))")

    # Her sütun için: tüm üründe dolu sayısı, Türkiye ürünlerinde dolu sayısı, örnek değer
    parcalar = ['count(*) AS "n|tum"', f'count(*) FILTER (WHERE {TR}) AS "n|tr"']
    for c in sutunlar:
        parcalar += [f'count("{c}") AS "{c}|tum"',
                     f'count("{c}") FILTER (WHERE {TR}) AS "{c}|tr"',
                     f'any_value("{c}") FILTER (WHERE {TR}) AS "{c}|ornek"']

    print("Hesaplanıyor... (birkaç dakika sürebilir)")
    satir = duckdb.sql(f"SELECT {', '.join(parcalar)} FROM {kaynak}").df().iloc[0]
    n_tum, n_tr = satir["n|tum"], satir["n|tr"]
    print(f"Tüm ürün: {n_tum:,}  ·  Türkiye ürünü: {n_tr:,}")

    doluluk = pd.DataFrame({
        "sutun":    sutunlar,
        "tum_%":    [round(100 * satir[f"{c}|tum"] / n_tum, 1) for c in sutunlar],
        "tr_%":     [round(100 * satir[f"{c}|tr"] / max(n_tr, 1), 1) for c in sutunlar],
        "tr_ornek": [str(satir[f"{c}|ornek"])[:80] for c in sutunlar],
    })
    doluluk.to_csv(DOLULUK, index=False)

pd.set_option("display.max_rows", 300)
pd.set_option("display.max_colwidth", 80)
doluluk.sort_values("tr_%", ascending=False)

**Bu tabloya bakarken düşünülecek sorular:**
- Türkiye ürünlerinde %30'un üstünde dolu olan ve projeye yeni bir özellik katabilecek sütun var mı?
- Görev A'da **etiket sızıntısı** yaratabilecek sütunlar hangileri? (Nutri-Score'la doğrudan ilişkili olanlar: besin değerleri, `nutrient_levels_tags`, `nova_group`, `pnns_groups_*`, `food_groups_*`...)
- Hangi sütunlar sağlık özeti katmanını zenginleştirebilir? (Örneğin `labels_tags`: organik, helal, glütensiz etiketleri.)

Yeni bir sütun kullanmaya karar verirsen, 3. adımdaki `SECILECEK` listesine ekle.

## 3. CSV → Parquet (hafifletilmiş)

**Neden?** CSV her açılışta baştan sona taranır (211 sütun, ~9 GB açık metin). Parquet ise sütun bazlı ve sıkıştırılmış bir biçimdir: yalnızca istenen sütunları okur. Bu dönüşümü **bir kez** yapıyoruz; sonraki bütün sorgular saniyeler sürer.

**Okuma ayarları:**
- `delim='\t'`: Dosya sekmeyle ayrılmış.
- `quote=''`: Tırnak işaretleri özel karakter sayılmaz. Ürün adlarındaki bozuk tırnaklar satırları karıştırmasın diye.
- `all_varchar=true`: Her şey önce metin olarak okunur. Barkodlardaki baştaki sıfırlar kaybolmaz.
- `ignore_errors=true`: Bozuk satırlar atlanır.

**Seçilen sütunlar ve kullanım amaçları:**

| Sütun grubu | Kullanım |
|---|---|
| `code`, `product_name`, `brands`, `brands_tags` | Kimlik, model girdisi, marka bazlı bölme |
| `countries_tags` | Türkiye / Avrupa grupları |
| `ingredients_text` | Model girdisi (Görev A, B, D) |
| `ingredients_tags`, `labels_tags` | Helal / vegan uygunluk analizi (sertifika etiketi + bileşen listesi) |
| `allergens`, `traces_tags`, `additives_tags` | Etiketler (Görev B, D) |
| `nutriscore_grade` | **Hedef değişken** (Görev A) |
| `image_*_url` | Görsel indirme |
| `*_100g`, `nutriscore_score`, `nova_group`, `categories_tags`... | ⚠️ **Model girdisi DEĞİL.** Yalnızca sağlık özeti katmanı, analiz ve zayıf etiket doğrulaması için |

> ⚠️ **Etiket sızıntısı uyarısı:** Nutri-Score besin değerlerinden (`*_100g`) doğrudan hesaplanır. Bu sütunları Görev A'da girdi olarak kullanmak modele cevabı vermek demektir.

Dosya zaten varsa ve bütün sütunları içeriyorsa bu adım atlanır.

In [ ]:
SECILECEK = '''
    code, product_name, brands, brands_tags, countries_tags,
    categories_tags, labels_tags, main_category,
    ingredients_text, ingredients_tags, ingredients_analysis_tags,
    allergens, traces_tags, additives_tags,
    TRY_CAST(additives_n AS INTEGER)                        AS additives_n,
    nutriscore_grade,
    TRY_CAST(nutriscore_score AS INTEGER)                   AS nutriscore_score,
    TRY_CAST(nova_group AS INTEGER)                         AS nova_group,
    TRY_CAST("energy-kcal_100g" AS DOUBLE)                  AS energy_kcal_100g,
    TRY_CAST(fat_100g AS DOUBLE)                            AS fat_100g,
    TRY_CAST("saturated-fat_100g" AS DOUBLE)                AS saturated_fat_100g,
    TRY_CAST(sugars_100g AS DOUBLE)                         AS sugars_100g,
    TRY_CAST(salt_100g AS DOUBLE)                           AS salt_100g,
    TRY_CAST(fiber_100g AS DOUBLE)                          AS fiber_100g,
    TRY_CAST(proteins_100g AS DOUBLE)                       AS proteins_100g,
    TRY_CAST("fruits-vegetables-legumes_100g" AS DOUBLE)    AS fruits_veg_legumes_100g,
    image_url, image_ingredients_url, image_nutrition_url
'''
# Listeye yeni sütun eklersen, mevcut dosyada eksik olduğu fark edilip dönüşüm yeniden yapılır
GEREKEN_SUTUNLAR = {p.split(" AS ")[-1].strip().strip('"')
                    for p in SECILECEK.split(",") if p.strip()}

def slim_hazir_mi():
    if not os.path.exists(SLIM):
        return False
    mevcut = set(duckdb.sql(f"DESCRIBE SELECT * FROM '{SLIM}'").df()["column_name"])
    return GEREKEN_SUTUNLAR <= mevcut

if slim_hazir_mi():
    print(f"Zaten var, atlanıyor: {SLIM}")
else:
    print("Dönüştürülüyor... (birkaç dakika sürebilir)")
    duckdb.sql(f'''
        COPY (
          SELECT {SECILECEK}
          FROM read_csv('{CSV}', delim='\t', quote='', header=true,
                        all_varchar=true, ignore_errors=true)
        ) TO '{SLIM}' (FORMAT parquet, COMPRESSION zstd)
    ''')
    print("Bitti.")

print(f"Boyut: {os.path.getsize(SLIM)/1e9:.2f} GB")
print("Toplam ürün:", duckdb.sql(f"SELECT count(*) FROM '{SLIM}'").fetchone()[0])

## 4. Türkiye ürünlerini işaretleme

`off` adında bir **görünüm (view)** tanımlıyoruz: dosyanın kopyası değil, her sorguda dosyayı okuyup üç yeni sütun ekleyen bir "kayıtlı sorgu".

| Sütun | Anlamı | Neden |
|---|---|---|
| `tr_ulke` | Ülke etiketi `en:turkey` **veya** `en:turkiye` | Veritabanında Türkiye için iki farklı etiket var (bkz. Adım 5.1) |
| `tr_barkod` | Barkod 868 veya 869 ile başlıyor | GS1'in Türkiye'ye ayırdığı önekler; Türk firmalarının ürünleri |
| `ns_var` | Nutri-Score a–e arasında | `unknown`, `not-applicable` gibi değerler hariç; boşsa "yok" sayılır |

`coalesce(x, '')` ve `coalesce(x, false)`: Boş değerleri boş metne ya da "yanlış"a çevirir. SQL'de boş değer (NULL) üzerinde yapılan karşılaştırmalar sonuç vermediği için gerekli.

In [ ]:
con = duckdb.connect()
con.sql(f'''
CREATE OR REPLACE VIEW off AS
SELECT *,
       regexp_matches(coalesce(countries_tags, ''), 'en:(turkey|turkiye)')  AS tr_ulke,
       regexp_matches(coalesce(code, ''), '^86[89]')                         AS tr_barkod,
       coalesce(nutriscore_grade IN ('a','b','c','d','e'), false)            AS ns_var
FROM '{SLIM}'
''')
con.sql("SELECT count(*) AS toplam, sum(tr_ulke::INT) AS tr_ulke, sum(tr_barkod::INT) AS tr_barkod FROM off").show()

## 5. Veri keşfi

### 5.1 Türkiye etiket varyantları

İlk denemede yalnızca `en:turkey` aranmış ve **601 ürün kaçırılmıştı**. Bu sorgu, adında "turk" geçen bütün ülke etiketlerini listeler. Beklenen sonuç: `en:turkey`, `en:turkiye` ve Türkiye dışı iki ülke (Turks ve Caicos Adaları, Türkmenistan).

**Bulgu (rapor):** Ülkenin uluslararası adının 2022'de değişmesiyle veritabanında iki farklı etiket oluşmuş.

`unnest(string_split(...))`: `en:france,en:turkey` gibi listeyi ayrı satırlara açar. DuckDB'de `unnest` ile `GROUP BY` aynı adımda kullanılamadığı için `WITH` (CTE) ile iki adıma bölünmüştür.

In [ ]:
con.sql('''
WITH acik AS (
  SELECT unnest(string_split(countries_tags, ',')) AS ulke FROM off
)
SELECT ulke, count(*) AS n
FROM acik
WHERE ulke ILIKE '%turk%'
GROUP BY ulke ORDER BY n DESC
''').show()

### 5.2 Türk barkodlu ürünler hangi ülkelerde kayıtlı?

Barkodu 868/869 olan ama Türkiye etiketi taşımayan ürünlerin ülke dağılımı.

**Bulgu (rapor):** Çoğunluk Fransa, Almanya, Irak, ABD ve Körfez ülkelerinde → **Türk ihracat ürünleri**. Etiketleri Fransızca, Almanca veya Arapça olabilir; bu yüzden ayrı bir grup (`TR_barkod`) olarak tutulacak ve dil tespiti yapılacak.

In [ ]:
ulke_dagilimi = con.sql('''
WITH acik AS (
  SELECT unnest(string_split(countries_tags, ',')) AS ulke
  FROM off
  WHERE tr_barkod AND NOT tr_ulke
)
SELECT ulke, count(*) AS n
FROM acik
GROUP BY ulke ORDER BY n DESC
LIMIT 15
''').df()
ulke_dagilimi.to_csv(f"{REPORTS}/tr_barkod_ulke_dagilimi.csv", index=False)
ulke_dagilimi

### 5.3 Eleme tablosu: Türkiye ürünleri hangi görevde kullanılabilir?

Her sütun bir koşulu (ya da koşulların kesişimini) sayar. Bu tablo raporun **"Veri ve Ön İşleme"** bölümüne girecek.

| Sütun | Anlamı |
|---|---|
| `metin_var` / `ic_gorsel_var` | İçindekiler metni / fotoğrafı olan ürünler |
| `A_metin_ile` | Görev A (Nutri-Score) için şu anki hâliyle kullanılabilir |
| `A_metin_veya_ocr` | Metni olmayanlara OCR uygularsak Görev A'ya katılabilecekler |
| `C_ocr_havuzu` | Hem fotoğrafı hem doğru metni olan → OCR modellerini sınamak için |
| `ocr_ile_doldurulacak` | Fotoğrafı var ama metni yok → OCR'ın kapattığı boşluk |
| `zayif_etiket_adayi` | Nutri-Score yok ama besin tablosu fotoğrafı var → OCR ile etiketlenebilir |

**Önceki bulgular:** İçindekiler fotoğrafı olan ürünler (≈6.300), metni olanlardan (≈3.900) fazla. ≈8.000 ürün zayıf etiketle Görev A'ya eklenebilir.

In [ ]:
eleme = con.sql('''
WITH tr AS (
  SELECT *,
         CASE WHEN tr_ulke THEN 'TR_ulke' ELSE 'TR_barkod' END                AS grup,
         ingredients_text IS NOT NULL                                          AS metin,
         image_ingredients_url IS NOT NULL                                     AS ic_gorsel,
         image_nutrition_url IS NOT NULL                                       AS besin_gorsel,
         image_url IS NOT NULL AND image_url NOT LIKE '%/invalid/%'            AS on_gorsel,
         product_name IS NOT NULL                                              AS ad
  FROM off WHERE tr_ulke OR tr_barkod
)
SELECT grup,
  count(*)                                                                     AS toplam,
  count(*) FILTER (WHERE metin)                                                AS metin_var,
  count(*) FILTER (WHERE ic_gorsel)                                            AS ic_gorsel_var,
  count(*) FILTER (WHERE ns_var)                                               AS ns_var,
  count(*) FILTER (WHERE ns_var AND ad AND on_gorsel AND metin)                AS A_metin_ile,
  count(*) FILTER (WHERE ns_var AND ad AND on_gorsel AND (metin OR ic_gorsel)) AS A_metin_veya_ocr,
  count(*) FILTER (WHERE metin AND ic_gorsel)                                  AS C_ocr_havuzu,
  count(*) FILTER (WHERE ic_gorsel AND NOT metin)                              AS ocr_ile_doldurulacak,
  count(*) FILTER (WHERE NOT ns_var AND besin_gorsel)                          AS zayif_etiket_adayi
FROM tr
GROUP BY grup ORDER BY grup
''').df()
eleme.to_csv(f"{REPORTS}/tr_eleme_tablosu.csv", index=False)
eleme

## 6. Proje veri kümesini oluşturma

Üç grup tek bir dosyada birleştirilir:

| Grup | Seçim kuralı | Neden |
|---|---|---|
| `TR_ulke` | Türkiye etiketi + (metin **veya** içindekiler fotoğrafı **veya** besin tablosu fotoğrafı) | Bütün görevler ve zayıf etiket kümesi için; hangi ürünün hangi görevde kullanılacağına sonra karar verilecek |
| `TR_barkod` | Barkod 868/869 + aynı koşullar | Türk üretimi ihracat ürünleri |
| `AB` | 6 Avrupa ülkesi + Görev A'nın bütün koşulları, **28.300 ürün** | Eğitim verisini büyütmek: Nutri-Score Avrupa kökenli; Türk Gıda Kodeksi AB etiketleme kurallarıyla uyumlu |

**Avrupa örneklemesi iki adımda yapılır:**
1. `ab_aday`: Önce bütün koşullar uygulanır (≈400 bin ürün).
2. `ab`: Sonra aralarından 28.300 tanesi seçilir.

> ⚠️ İlk denemede `USING SAMPLE` kullanılmıştı. DuckDB bu ifadeyi `WHERE` filtresinden **önce** çalıştırdığı için tüm veritabanından 28.300 ürün seçilip sonra filtrelenmiş ve yalnızca 1.931 ürün kalmıştı.

**Tekrar üretilebilir rastgele seçim:** `ORDER BY hash(code || '-42')` her barkoda tohum değeriyle karışık bir numara verir; ilk 28.300'ü alınır. Sonuç rastgele ama her çalıştırmada **aynıdır**.

**Tekilleştirme:** Veritabanında aynı barkod birden fazla satırda geçebiliyor (AB adaylarında 406.732 satır, 406.716 farklı barkod). `QUALIFY row_number() OVER (PARTITION BY code ...) = 1` her barkoddan yalnızca en eksiksiz satırı tutar. Bu işlem **seçimden önce** yapılır.

> ⚠️ İlk sürümde tekilleştirme seçimden **sonra** yapılıyordu: seçilen 28.300 satırın içindeki tekrarlar silinince AB sayısı 28.300'ün altına düşüyordu. 7. adımdaki kontrol bu hatayı yakaladı.

In [ ]:
ulke_regex = "en:(" + "|".join(AB_ULKELER) + ")"

ab_aday_sayisi = con.sql(f'''
SELECT count(DISTINCT code) FROM off
WHERE NOT (tr_ulke OR tr_barkod)
  AND regexp_matches(coalesce(countries_tags, ''), '{ulke_regex}')
  AND ns_var AND product_name IS NOT NULL AND ingredients_text IS NOT NULL
  AND image_url IS NOT NULL AND image_url NOT LIKE '%/invalid/%'
''').fetchone()[0]
print(f"AB aday sayısı: {ab_aday_sayisi:,}")

con.sql(f'''
COPY (
  WITH temel AS (
    SELECT *,
           ingredients_text IS NOT NULL                                AS metin,
           image_ingredients_url IS NOT NULL                           AS ic_gorsel,
           image_nutrition_url IS NOT NULL                             AS besin_gorsel,
           image_url IS NOT NULL AND image_url NOT LIKE '%/invalid/%'  AS on_gorsel,
           product_name IS NOT NULL                                    AS ad
    FROM off
    -- Aynı barkod birden fazla satırda geçiyorsa en eksiksiz olanı tut (seçimden ÖNCE)
    QUALIFY row_number() OVER (
        PARTITION BY code
        ORDER BY ns_var DESC, (ingredients_text IS NOT NULL) DESC,
                 (image_url IS NOT NULL) DESC, product_name, ingredients_text
    ) = 1
  ),
  tr AS (
    SELECT *, CASE WHEN tr_ulke THEN 'TR_ulke' ELSE 'TR_barkod' END AS grup
    FROM temel
    WHERE (tr_ulke OR tr_barkod) AND (metin OR ic_gorsel OR besin_gorsel)
  ),
  ab_aday AS (                                    -- 1. adım: önce filtrele
    SELECT *, 'AB' AS grup
    FROM temel
    WHERE NOT (tr_ulke OR tr_barkod)
      AND regexp_matches(coalesce(countries_tags, ''), '{ulke_regex}')
      AND ns_var AND ad AND on_gorsel AND metin
  ),
  ab AS (                                         -- 2. adım: sonra seç
    SELECT * FROM ab_aday
    ORDER BY hash(code || '-{SEED}'), code
    LIMIT {N_AB}
  )
  SELECT * FROM tr
  UNION ALL
  SELECT * FROM ab
) TO '{DS}' (FORMAT parquet)
''')
print("Kaydedildi:", DS)

## 7. Kontroller

Böyle bir hatanın (örneğin AB'den 1.931 ürün gelmesi) gözden kaçmaması için sonuç otomatik olarak kontrol edilir. `assert` koşul sağlanmazsa hata verir ve not defteri durur.

Beklenen değerler:
- AB satırı tam olarak **28.300**
- Barkodlar tekil
- Bütün AB ürünlerinin Nutri-Score'u var

In [ ]:
df = pd.read_parquet(DS)

print(f"Toplam ürün: {len(df):,}\n")
print(df.grup.value_counts(), "\n")
ozet = df.groupby("grup")[["ns_var", "metin", "ic_gorsel", "besin_gorsel", "on_gorsel"]].sum()
ozet.to_csv(f"{REPORTS}/dataset_grup_ozeti.csv")
display(ozet)

assert (df.grup == "AB").sum() == min(N_AB, ab_aday_sayisi), "AB örnek sayısı beklenenden farklı!"
assert df.code.is_unique, "Tekrarlanan barkod var!"
assert df.loc[df.grup == "AB", "ns_var"].all(), "Nutri-Score'u olmayan AB ürünü var!"
print("✅ Bütün kontroller geçti.")

### Nutri-Score sınıf dağılımı (gruplara göre)

Sınıf dengesizliğini görmek için. Görev A'da birincil ölçütün **makro-F1** olmasının sebebi bu: az örnekli sınıflardaki başarımı da eşit ağırlıkla hesaba katar.

In [ ]:
dagilim = (df[df.ns_var]
           .groupby(["grup", "nutriscore_grade"]).size()
           .unstack(fill_value=0))
dagilim.to_csv(f"{REPORTS}/nutriscore_dagilimi.csv")
display(dagilim)
display((dagilim.div(dagilim.sum(axis=1), axis=0) * 100).round(1))

## 8. Eski dosyaların temizliği

İlk denemelerden kalan dosyalar:
- `subset.parquet`: İlk pandas denemesi; `en:turkiye` ve barkod öneki ürünleri eksik → **kullanılmayacak**.
- `products.parquet`: Bütün 211 sütunu içeren dönüşüm; `off_slim.parquet` ile gereksiz hâle geldi ve Drive'da çok yer kaplar.

Önce listelenir. Silmek için `SIL = True` yap ve hücreyi yeniden çalıştır.

In [ ]:
SIL = False

for ad in ["subset.parquet", "products.parquet"]:
    yol = f"{BASE}/{ad}"
    if os.path.exists(yol):
        print(f"{ad}: {os.path.getsize(yol)/1e9:.2f} GB", end="  ")
        if SIL:
            os.remove(yol); print("→ silindi")
        else:
            print("→ silmek için SIL = True")
    else:
        print(f"{ad}: yok")

print("\nDrive klasöründeki dosyalar:")
for ad in sorted(os.listdir(BASE)):
    yol = f"{BASE}/{ad}"
    if os.path.isfile(yol):
        print(f"  {ad:30s} {os.path.getsize(yol)/1e6:10.1f} MB")

## Özet ve sonraki adım

**Bu not defterinde:**
- 4,5 milyon ürünlük döküm, gerekli sütunlarla Parquet'e çevrildi.
- Türkiye ürünleri iki yöntemle bulundu (ülke etiketi + barkod öneki); `en:turkiye` varyantı düzeltildi.
- Türkiye + 28.300 Avrupa ürününden oluşan `dataset.parquet` oluşturuldu ve kontrol edildi.
- Rapor tabloları `reports/` klasörüne kaydedildi.

**Sonraki not defteri — `02_dil_ve_bolme.ipynb`:**
1. İçindekiler metinlerinde dil tespiti (fastText)
2. Marka bazlı train / val / test bölmesi → `split.csv`